# Customer Churn Prediction

## Objetivo del proyecto

El objetivo de este proyecto es analizar los factores asociados al abandono de clientes (*customer churn*) en una empresa de telecomunicaciones y desarrollar un modelo predictivo capaz de identificar clientes con alto riesgo de cancelación del servicio.

Identificar el riesgo de churn de forma temprana ayuda a retener clientes clave y mitigar el impacto financiero por cancelaciones.

A lo largo del proyecto se realizará:

- Comprensión y limpieza de datos.
- Análisis exploratorio de variables.
- Preprocesamiento y transformación de datos.
- Entrenamiento y evaluación de modelos de machine learning.
- Interpretación de resultados y conclusiones de negocio.

__Importamos librerías.__

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
pd.set_option('display.max_columns', None)

__Cargamos el dataset y análisis de la estructura.__

In [2]:
df = pd.read_csv(r"..\Data\Raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
print(df.shape)

(7043, 21)


__Visualizamos las columnas del dataset.__

In [4]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

__Información general del dataset__

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


__Primeras filas del dataset__

In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Observaciones iniciales del dataset

El dataset contiene información demográfica, contractual y de consumo de clientes de una empresa de telecomunicaciones (Telco). Se dispone de 7.043 registros y 21 variables, incluyendo la variable objetivo `Churn`, que indica si el cliente abandonó el servicio.

La mayoría de variables son categóricas (`object`), especialmente aquellas relacionadas con servicios contratados, métodos de pago y características del cliente. También se observan variables numéricas relevantes como:

- `tenure`: antigüedad del cliente.
- `MonthlyCharges`: importe mensual facturado.
- `TotalCharges`: gasto total acumulado.

Mediante `info()` no se detectan valores nulos en una primera revisión. Sin embargo, la variable `TotalCharges` aparece como tipo `object` cuando debería ser numérica, lo que sugiere la existencia de posibles inconsistencias o valores vacíos que deberán analizarse durante la fase de limpieza de datos.

`customerID` actúa solo como identificador único del cliente y probablemente no aporte valor predictivo al modelo.

# Estadísticas descriptivas.

## Variables numéricas.

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


# Análisis de variables categóricas.

## Variables categóricas del dataset.

In [8]:
categorical_columns = df.select_dtypes(include='object').columns

categorical_columns

Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges',
       'Churn'],
      dtype='object')

## Revisión de categorías.

In [9]:
for col in categorical_columns:
    
    print(f"\n{'='*60}")
    print(f"Variable: {col}")
    print(f"{'='*60}")
    
    print(df[col].value_counts())


Variable: customerID
customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
             ..
6713-OKOMC    1
1452-KIOVK    1
9305-CDSKC    1
9237-HQITU    1
7795-CFOCW    1
Name: count, Length: 7043, dtype: int64

Variable: gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Variable: Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Variable: Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

Variable: PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

Variable: MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

Variable: InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

Variable: OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

Va

# Análisis de valores nulos.

In [10]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [11]:
df['TotalCharges'].unique()

array(['29.85', '1889.5', '108.15', ..., '346.45', '306.6', '6844.5'],
      dtype=object)

`TotalCharges` aparece como tipo object, aunque representa una variable numérica (importe total facturado a cada cliente), lo que sugiere la existencia de valores inconsistentes o vacíos que requieren limpieza previa al análisis.

Para ello convertimos los valores de la columna a formato numérico, transformando en NaN aquellos valores que no pueden convertirse correctamente.

In [12]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')

In [13]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

## Análisis de valores nulos en TotalCharges

In [14]:
df[df['TotalCharges'].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


## Tratamiento de valores nulos

Dado que los valores nulos corresponden a clientes con antigüedad 0 meses, se decide imputar estos valores a 0, ya que no han generado facturación acumulada.

In [15]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [16]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Eliminación de variables no informativas

La variable `customerID` se elimina del análisis ya que actúa únicamente como identificador único y no aporta información predictiva al modelo.

In [17]:
df = df.drop('customerID', axis=1)

## Transformación de la variable objetivo

La variable objetivo `Churn` se transforma a formato binario para su uso en modelos de machine learning:

- Yes → 1
- No → 0

In [18]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

In [19]:
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

## Verificación final del dataset

Tras la fase de comprensión y limpieza inicial, se revisa nuevamente la estructura del dataset para confirmar que:

- los tipos de datos son correctos,
- no existen valores nulos pendientes,
- y las variables están preparadas para el análisis exploratorio.

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [21]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Dimensión final del dataset

Tras la fase inicial de limpieza, el dataset mantiene la totalidad de registros originales y queda preparado para la fase de análisis exploratorio.

In [22]:
df.shape

(7043, 20)

## Guardado del dataset tras la fase de limpieza inicial para asegurar trazabilidad en el pipeline.

Guardamos una versión intermedia del dataset tras la fase inicial de limpieza y comprensión de datos para mantener la trazabilidad y reutilizar los datos procesados en las siguientes etapas del proyecto.

In [23]:
os.makedirs(r"..\Data\Interim", exist_ok=True)

df.to_csv(
    r"..\Data\Interim\telco_churn_cleaned_step1.csv",
    index=False
)

## Conclusiones

En esta primera fase del proyecto se realizó una comprensión general del dataset y una limpieza inicial de los datos.

Se identificaron inconsistencias en la variable `TotalCharges`, se corrigieron tipos de datos, se trataron valores faltantes y se preparó la variable objetivo para futuras etapas de modelado.

El dataset queda listo para realizar el análisis exploratorio de datos (EDA), donde se investigarán patrones y relaciones asociadas al abandono de clientes.